In [2]:
import sys
import os
import tifffile
import numpy as np
import cupy as cp
from pathlib import Path
import tifffile
import matplotlib.pyplot as plt

sys.path.append('/home/cyf/wbi/Virginia/code_for_paper/wholistic_registration/src/wholistic_registration')
current_dir = os.path.dirname(os.path.abspath("__file__"))
root_dir = os.path.dirname(current_dir)
sys.path.append(root_dir)
from utils.IO import readND2Frame, readMeta,readTiff
from utils.simulation import generateMotion_Biophysical
from utils import calFlow3d_Wei_v1, calFlowCrossResolution
from utils import visualization as vis
from utils import preprocess as prep
from utils import option

CuPy is available with CUDA - using GPU acceleration


In [3]:
F260515_mov ,_= readTiff("/home/cyf/wbi/Virginia/raw_data/f260517/260517_exp_00001_TZCYX.ome.tiff")
F260515_ref,_ = readTiff("/home/cyf/wbi/Virginia/raw_data/f260517/260517_anat_00003_TZCYX.ome.tiff")

In [1]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt


def compute_global_normalization_range(
    img,
    slice_indices=None,
    index_base=0,
    pmin=1,
    pmax=99,
):
    if slice_indices is None:
        data = img
    else:
        z_indices = [idx - index_base for idx in slice_indices]
        data = img[z_indices]

    vmin, vmax = np.percentile(data, [pmin, pmax])
    return float(vmin), float(vmax)


def normalize_to_float01(slice_img, vmin, vmax):
    slice_img = slice_img.astype(np.float32)
    slice_img = np.clip(slice_img, vmin, vmax)

    if vmax > vmin:
        slice_img = (slice_img - vmin) / (vmax - vmin)
    else:
        slice_img = np.zeros_like(slice_img)

    return slice_img


def apply_colormap(slice_float01, cmap_name="hot"):
    """
    Convert normalized 2D image in [0, 1] to RGB uint8 image.

    Common cmap_name options:
        "gray"
        "hot"       similar to FIJI Fire
        "inferno"
        "magma"
        "viridis"
        "plasma"
        "turbo"
        "jet"
    """

    cmap = plt.get_cmap(cmap_name)
    rgba = cmap(slice_float01)          # shape: (H, W, 4), float in [0, 1]
    rgb = rgba[..., :3]                 # remove alpha channel
    rgb = (rgb * 255).astype(np.uint8)

    return rgb


def save_selected_slices_colored_global_norm(
    img,
    output_dir,
    slice_indices,
    index_base=0,
    normalize_scope="volume",
    pmin=1,
    pmax=99,
    cmap_name="hot",
    prefix="slice",
):
    """
    Save selected slices as colored PNG with global normalization.
    """

    os.makedirs(output_dir, exist_ok=True)

    if img.ndim != 3:
        raise ValueError(f"Expected img shape (Z, Y, X), but got {img.shape}")

    if normalize_scope == "volume":
        vmin, vmax = compute_global_normalization_range(
            img,
            slice_indices=None,
            pmin=pmin,
            pmax=pmax,
        )
    elif normalize_scope == "selected":
        vmin, vmax = compute_global_normalization_range(
            img,
            slice_indices=slice_indices,
            index_base=index_base,
            pmin=pmin,
            pmax=pmax,
        )
    else:
        raise ValueError("normalize_scope should be 'volume' or 'selected'.")

    print(f"Global normalization range: vmin={vmin:.3f}, vmax={vmax:.3f}")
    print(f"Colormap: {cmap_name}")

    Z = img.shape[0]

    for idx in slice_indices:
        z = idx - index_base

        if z < 0 or z >= Z:
            print(f"Skip slice {idx}: out of range.")
            continue

        slice_img = img[z]

        slice_float01 = normalize_to_float01(slice_img, vmin, vmax)
        rgb = apply_colormap(slice_float01, cmap_name=cmap_name)

        save_path = os.path.join(output_dir, f"{prefix}_{idx:04d}_{cmap_name}.png")
        Image.fromarray(rgb).save(save_path)

        print(f"Saved: {save_path}")

In [22]:
slice_indices = [0,3,6,9,12,15,18]

save_selected_slices_colored_global_norm(
    img=F260515_mov[0,:,1,:,:],
    output_dir="Fig1.A_refStack",
    slice_indices=slice_indices,
    index_base=1,
    normalize_scope="volume",
    pmin=1,
    pmax=99,
    cmap_name="turbo",       # 类似 FIJI 的 Fire
    prefix="fire"
)

Global normalization range: vmin=-157.000, vmax=5061.000
Colormap: turbo
Skip slice 0: out of range.
Saved: Fig1.A_refStack/fire_0003_turbo.png
Saved: Fig1.A_refStack/fire_0006_turbo.png
Saved: Fig1.A_refStack/fire_0009_turbo.png
Saved: Fig1.A_refStack/fire_0012_turbo.png
Saved: Fig1.A_refStack/fire_0015_turbo.png
Saved: Fig1.A_refStack/fire_0018_turbo.png


In [ ]:
cmap_name = "gray"      # 灰度
cmap_name = "hot"       # 类似 FIJI Fire
cmap_name = "inferno"   # 黑-紫-橙-黄，很适合荧光图
cmap_name = "magma"     # 黑-紫-粉-黄
cmap_name = "viridis"   # 蓝绿黄
cmap_name = "plasma"    # 紫黄
cmap_name = "turbo"     # 类似彩虹，但比 jet 更现代
cmap_name = "jet"       # 老式彩虹色，不太推荐论文使用

In [12]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# 自定义颜色：深蓝 #102060 → 粉红 #FF6384
start_color = "#102060"
end_color = "#FF6384"

# 创建渐变色彩
cmap = mcolors.LinearSegmentedColormap.from_list(
    "custom_gradient",
    [start_color, end_color],
    N=256
)

# 创建画布：竖版长条
fig, ax = plt.subplots(figsize=(2, 10))  # 宽, 高

norm = mcolors.Normalize(vmin=0, vmax=1)

# 绘制竖向颜色条
cbar = plt.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap),
    cax=ax,
    orientation="vertical"
)

# 不显示任何标注、刻度和刻度线
cbar.set_ticks([])
cbar.set_label("")
cbar.ax.tick_params(
    left=False,
    right=False,
    labelleft=False,
    labelright=False,
    length=0
)

# 可选：保留或去掉外边框
# 保留细边框：
cbar.outline.set_linewidth(1.0)

# 如果不要边框，改用这一行：
# cbar.outline.set_visible(False)

plt.tight_layout()

# 输出 PDF 文件
plt.savefig("colorbar_vertical.pdf", dpi=300, bbox_inches="tight", transparent=True)
plt.close()

print("✅ 已生成 PDF 文件：colorbar_vertical.pdf")

✅ 已生成 PDF 文件：colorbar_vertical.pdf


In [16]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter


def _normalize_volume(img, normalize_scope="volume", pmin=1, pmax=99):
    """
    img: (Z, Y, X)
    return normalized img in [0,1]
    """
    img = np.asarray(img, dtype=np.float32)

    if normalize_scope == "volume":
        lo = np.percentile(img, pmin)
        hi = np.percentile(img, pmax)
        out = np.clip((img - lo) / (hi - lo + 1e-8), 0, 1)
    elif normalize_scope == "slice":
        out = np.empty_like(img, dtype=np.float32)
        for z in range(img.shape[0]):
            lo = np.percentile(img[z], pmin)
            hi = np.percentile(img[z], pmax)
            out[z] = np.clip((img[z] - lo) / (hi - lo + 1e-8), 0, 1)
    else:
        raise ValueError("normalize_scope must be 'volume' or 'slice'")

    return out


def generate_smooth_motion_field_3d(
    shape_zyx,
    amp=12.0,
    sigma_yx=30.0,
    sigma_z=1.0,
    seed=0,
):
    """
    Generate a smooth synthetic motion field for illustration.

    Parameters
    ----------
    shape_zyx : tuple
        (Z, Y, X)
    amp : float
        Approximate max arrow length in pixels
    sigma_yx : float
        Spatial smoothing in XY
    sigma_z : float
        Smoothing along Z
    seed : int
        Random seed

    Returns
    -------
    motion : ndarray
        shape (Z, Y, X, 2), where motion[...,0]=dx, motion[...,1]=dy
    """
    z, y, x = shape_zyx
    rng = np.random.default_rng(seed)

    dx = rng.normal(size=(z, y, x)).astype(np.float32)
    dy = rng.normal(size=(z, y, x)).astype(np.float32)

    dx = gaussian_filter(dx, sigma=(sigma_z, sigma_yx, sigma_yx))
    dy = gaussian_filter(dy, sigma=(sigma_z, sigma_yx, sigma_yx))

    mag = np.sqrt(dx**2 + dy**2)
    ref = np.percentile(mag, 99)

    if ref < 1e-8:
        ref = 1.0

    scale = amp / ref
    dx *= scale
    dy *= scale

    motion = np.stack([dx, dy], axis=-1)  # (Z, Y, X, 2)
    return motion


def save_selected_slices_with_motion_arrows(
    img,
    output_dir,
    slice_indices,
    index_base=1,
    normalize_scope="volume",
    pmin=1,
    pmax=99,
    cmap_name="turbo",
    prefix="fire_motion",
    motion_field=None,
    grid_step=40,
    arrow_color="cyan",
    arrow_width=0.005,
    arrow_scale=1,
    point_color="yellow",
    point_size=8,
    amp=12.0,
    sigma_yx=30.0,
    sigma_z=1.0,
    seed=0,
    dpi=200,
):
    """
    Save selected slices with sparse motion arrows overlaid.

    Parameters
    ----------
    img : ndarray
        3D image, shape (Z, Y, X)
    output_dir : str
        Output folder
    slice_indices : list[int]
        Which z slices to save
    motion_field : ndarray or None
        If provided, should be shape (Z, Y, X, 2)
        motion[...,0] = dx, motion[...,1] = dy
        If None, a smooth synthetic field will be generated.
    grid_step : int
        Sparse grid spacing in pixels
    arrow_scale : float or None
        If None, use actual pixel displacement directly.
        If not None, divide vector lengths by this value in quiver.
    """
    os.makedirs(output_dir, exist_ok=True)

    img = np.asarray(img)
    assert img.ndim == 3, f"Expected img shape (Z,Y,X), got {img.shape}"

    zdim, h, w = img.shape

    # normalize image
    img_norm = _normalize_volume(img, normalize_scope=normalize_scope, pmin=pmin, pmax=pmax)

    # motion field
    if motion_field is None:
        motion_field = generate_smooth_motion_field_3d(
            shape_zyx=img.shape,
            amp=amp,
            sigma_yx=sigma_yx,
            sigma_z=sigma_z,
            seed=seed,
        )

    motion_field = np.asarray(motion_field)
    assert motion_field.shape == (zdim, h, w, 2), \
        f"motion_field should have shape {(zdim,h,w,2)}, but got {motion_field.shape}"

    cmap = plt.get_cmap(cmap_name)

    # sparse grid
    ys = np.arange(grid_step // 2, h, grid_step)
    xs = np.arange(grid_step // 2, w, grid_step)
    XX, YY = np.meshgrid(xs, ys)

    for z in slice_indices:
        if z < 0 or z >= zdim:
            print(f"[Warning] slice index {z} out of range, skipped.")
            continue

        slice_gray = img_norm[z]
        slice_rgb = cmap(slice_gray)[..., :3]

        # sample motion on sparse grid
        dx = motion_field[z, YY, XX, 0]
        dy = motion_field[z, YY, XX, 1]

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(slice_rgb, origin="upper")

        # points = coarse locations
        ax.scatter(
            XX.ravel(),
            YY.ravel(),
            s=point_size,
            c=point_color,
            alpha=0.8,
            linewidths=0,
        )

        # arrows = coarse motion / coordinate displacement
        ax.quiver(
            XX,
            YY,
            dx,
            dy,
            color=arrow_color,
            angles="xy",
            scale_units="xy",
            scale=arrow_scale,   # None means literal data scale
            width=arrow_width,
            headwidth=4,
            headlength=5,
            headaxislength=4.5,
        )

        # ax.set_title(f"Slice {z + index_base}")
        ax.set_axis_off()
        plt.tight_layout()

        save_path = os.path.join(output_dir, f"{prefix}_slice_{z + index_base:03d}.png")
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight", pad_inches=0.02)
        plt.close(fig)

        print(f"Saved: {save_path}")

# slice_indices = [0, 3, 6, 9, 12, 15, 18]
slice_indices = [2, 5, 8, 11, 14, 17]
save_selected_slices_with_motion_arrows(
    img=F260515_mov[0, :, 1, :, :],   # (Z, Y, X)
    output_dir="Fig1.A_refStack_motion",
    slice_indices=slice_indices,
    index_base=1,
    normalize_scope="volume",
    pmin=1,
    pmax=99,
    cmap_name="inferno",
    prefix="fire_motion",
    arrow_width = 0.02,
    motion_field=None,   # None = 自动生成平滑示意 motion
    grid_step=130,        # 稀疏一些，更像 coarse coordinates
    arrow_color="white",
    point_color="white",
    amp=150.0,            # 控制箭头长度
    sigma_yx=200.0,       # 越大越平滑
    sigma_z=1.2,
    seed=20260527,
    dpi=300,
)




Saved: Fig1.A_refStack_motion/fire_motion_slice_003.png
Saved: Fig1.A_refStack_motion/fire_motion_slice_006.png
Saved: Fig1.A_refStack_motion/fire_motion_slice_009.png
Saved: Fig1.A_refStack_motion/fire_motion_slice_012.png
Saved: Fig1.A_refStack_motion/fire_motion_slice_015.png
Saved: Fig1.A_refStack_motion/fire_motion_slice_018.png
